# 01 — Exploratory Data Analysis

**Project:** Reddit Hyperlink GNN — edge sign classification.  
**Author:** Олександр Щепанчук (group ПМПм-12).

Every example below is an **observed** subreddit-to-subreddit hyperlink. Label 0 means *negative sentiment*, label 1 means *neutral/positive*. We never sample non-edges and we never treat label 0 as a non-edge.

This notebook loads the processed parquet (run `make data` first), produces all EDA figures, and writes a one-row stats summary to `reports/tables/stats_summary.csv`. Every figure cell is preceded by a short markdown explanation.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from reddit_gnn.analysis.graph_stats import (
    compute_basic_stats,
    compute_component_stats,
    compute_degree_stats,
    compute_reciprocity_stats,
)
from reddit_gnn.analysis.signed_stats import (
    compute_label_stats,
    negative_ratio_by_source,
    negative_ratio_by_target,
    signed_triad_counts,
)
from reddit_gnn.analysis.temporal_stats import edges_over_time, negative_ratio_over_time
from reddit_gnn.paths import PATHS
from reddit_gnn.visualization.distributions import (
    plot_degree_distribution,
    plot_label_distribution,
    plot_top_subreddits_by_degree,
)
from reddit_gnn.visualization.subgraphs import (
    plot_ego_signed_subgraph,
    plot_sampled_signed_subgraph,
)
from reddit_gnn.visualization.temporal import (
    plot_edges_over_time,
    plot_negative_ratio_over_time,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Load the processed dataset

Loads `data/processed/edges.parquet` (built by `scripts/prepare_data.py`) and the JSON sidecars.

In [ ]:
EDGES_PATH = PATHS.data_processed / 'edges.parquet'
META_PATH = PATHS.data_processed / 'metadata.json'
assert EDGES_PATH.exists(), f'Missing {EDGES_PATH}; run `make data` first.'

df = pd.read_parquet(EDGES_PATH)
metadata = json.loads(META_PATH.read_text())
print(f'Loaded {len(df):,} edges, {metadata["num_nodes"]:,} nodes')
df.head()

## 2. Basic graph statistics

Node / edge counts, density, degree summary, sanity checks (self-loops must be zero after `clean_edges`).

In [ ]:
basic = compute_basic_stats(df)
pd.DataFrame.from_records([basic]).T.rename(columns={0: 'value'})

## 3. Edge label distribution

Class balance over the two trained labels (0 = negative, 1 = neutral/positive). Sentiment in social-graph data is typically very skewed toward neutral/positive — this motivates `class_weight: balanced` and macro-F1 as the primary metric.

In [ ]:
label_stats = compute_label_stats(df)
print(label_stats)
fig = plot_label_distribution(df, save_path=PATHS.reports_figures / '01_label_distribution.png')
plt.show()

## 4. Degree distribution

We plot the **total** degree (in + out) histogram on a log-log axis. Heavy-tailed distributions are the norm for Reddit-style graphs: a few hub subreddits attract orders-of-magnitude more activity than the median.

In [ ]:
deg = compute_degree_stats(df)
print('summary:', deg['summary'])
fig = plot_degree_distribution(
    deg['total_degree'],
    degree_type='total',
    log_scale=True,
    save_path=PATHS.reports_figures / '01_degree_total_log.png',
)
plt.show()
fig = plot_degree_distribution(
    deg['in_degree'],
    degree_type='in',
    log_scale=True,
    save_path=PATHS.reports_figures / '01_degree_in_log.png',
)
plt.show()
fig = plot_degree_distribution(
    deg['out_degree'],
    degree_type='out',
    log_scale=True,
    save_path=PATHS.reports_figures / '01_degree_out_log.png',
)
plt.show()

## 5. Top subreddits by degree

Lists the most-connected subreddits as ranked by total, in, and out degree. These are the hub nodes that will dominate any neighbor-sampling regime.

In [ ]:
fig = plot_top_subreddits_by_degree(
    df, top_k=20, mode='total',
    save_path=PATHS.reports_figures / '01_top_total_degree.png',
)
plt.show()
fig = plot_top_subreddits_by_degree(
    df, top_k=20, mode='in',
    save_path=PATHS.reports_figures / '01_top_in_degree.png',
)
plt.show()
fig = plot_top_subreddits_by_degree(
    df, top_k=20, mode='out',
    save_path=PATHS.reports_figures / '01_top_out_degree.png',
)
plt.show()

## 6. Reciprocity

Reciprocity = fraction of directed edges `(u, v)` such that the reverse edge `(v, u)` also exists in the graph. High reciprocity argues for considering the graph effectively undirected for some message-passing schemes; low reciprocity argues for keeping direction.

In [ ]:
recip = compute_reciprocity_stats(df)
print(recip)

## 7. Weakly and strongly connected components

Computed via `scipy.sparse.csgraph.connected_components`. A small SCC count with a large dominant SCC indicates a rich back-and-forth core; many small SCCs indicate hub-and-spoke topology.

In [ ]:
comp = compute_component_stats(df)
print(comp)

## 8. Edges over time

Volume per month, broken down by sentiment. This visualizes the SNAP collection period (2014-01 through 2017-04) and motivates a **temporal** train/val/test split rather than a random split.

In [ ]:
fig = plot_edges_over_time(
    df, freq='ME',
    save_path=PATHS.reports_figures / '01_edges_over_time.png',
)
plt.show()

## 9. Negative-label share over time

Does the fraction of negative hyperlinks drift over time? Drift would mean the temporal split is the realistic evaluation regime; stationarity would let random splits stand in.

In [ ]:
fig = plot_negative_ratio_over_time(
    df, freq='ME',
    save_path=PATHS.reports_figures / '01_negative_ratio_over_time.png',
)
plt.show()

## 10. Negative ratios by source / target

Per-subreddit negativity rankings — which subreddits post the most negative hyperlinks (source), and which receive them (target).

In [ ]:
src_neg = negative_ratio_by_source(df, top_k=20)
tgt_neg = negative_ratio_by_target(df, top_k=20)
print('--- top-20 sources by negative-out ratio ---')
display(src_neg)
print('--- top-20 targets by negative-in ratio ---')
display(tgt_neg)

## 11. Signed triangles (sampled)

Counts of balanced and unbalanced triangles on a 50k-edge random sample. Under structural balance theory (Heider, Cartwright-Harary), balanced triangles should dominate. The sample keeps NetworkX triangle enumeration tractable.

In [ ]:
triads = signed_triad_counts(df, sample_cap=50_000)
print(triads)

## 12. Sampled signed subgraph

A random 300-edge subgraph drawn with red = negative, green = neutral/positive, arrows for direction. Purely qualitative — useful for sanity-checking that the cleaned graph looks sensible.

In [ ]:
fig = plot_sampled_signed_subgraph(
    df, max_edges=300, seed=42,
    save_path=PATHS.reports_figures / '01_sampled_signed_subgraph.png',
)
plt.show()

## 13. Ego subgraph of the highest-degree subreddit

1-hop ego graph around the most-connected subreddit, capped at 300 edges. The user can re-run this cell on any subreddit they're curious about by editing `ego_subreddit` below.

In [ ]:
totals = df.groupby('source_subreddit_norm').size().add(
    df.groupby('target_subreddit_norm').size(), fill_value=0,
)
ego_subreddit = totals.sort_values(ascending=False).index[0]
print('ego subreddit:', ego_subreddit)
fig = plot_ego_signed_subgraph(
    df, subreddit=ego_subreddit, radius=1, max_edges=300, seed=42,
    save_path=PATHS.reports_figures / '01_ego_signed_subgraph.png',
)
plt.show()

## 14. Stats summary table

Persists a single-row summary table to `reports/tables/stats_summary.csv`. This is the canonical place to look up dataset-level numbers when writing the report.

In [ ]:
summary = {
    'num_nodes': basic['num_nodes'],
    'num_edges': basic['num_edges'],
    'density': basic['density'],
    'avg_in_degree': basic['avg_in_degree'],
    'avg_out_degree': basic['avg_out_degree'],
    'max_in_degree': basic['max_in_degree'],
    'max_out_degree': basic['max_out_degree'],
    'self_loop_count': basic['self_loop_count'],
    'duplicate_edge_count': basic['duplicate_edge_count'],
    'num_positive': label_stats['num_positive'],
    'num_negative': label_stats['num_negative'],
    'positive_ratio': label_stats['positive_ratio'],
    'negative_ratio': label_stats['negative_ratio'],
    'reciprocity': recip['reciprocity'],
    'wcc_count': comp['wcc_count'],
    'largest_wcc_size': comp['largest_wcc_size'],
    'scc_count': comp['scc_count'],
    'largest_scc_size': comp['largest_scc_size'],
    'triads_balanced': triads['balanced'],
    'triads_unbalanced': triads['unbalanced'],
    'time_min': metadata['timestamp_range']['min'],
    'time_max': metadata['timestamp_range']['max'],
}
table_path = PATHS.reports_tables / 'stats_summary.csv'
table_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame([summary]).to_csv(table_path, index=False)
print(f'Wrote {table_path}')
pd.DataFrame([summary]).T.rename(columns={0: 'value'})